# Google Agent Development Kit (ADK): New-Hire Onboarding Agent
## End-to-End Deep-Dive Colab Notebook

**Source**: [GoogleCloudPlatform/generative-ai — agents/adk/new-hire-onboarding](https://github.com/GoogleCloudPlatform/generative-ai/tree/main/agents/adk/new-hire-onboarding)  
**License**: Apache 2.0 © 2026 Google LLC

---

## What You Will Learn

| # | Topic | Key Concept |
|---|-------|-------------|
| 1 | **State Schema** | Enum-based state machines vs raw JSON memory |
| 2 | **ADK Tools** | `ToolContext` pattern — how tools read/write session state |
| 3 | **Resume Handler** | Webhook-driven agent resumption (`runner.run_async + state_delta`) |
| 4 | **Multi-Agent Design** | Root agent delegates IT provisioning to a sub-agent |
| 5 | **Before-Agent Callback** | Idempotent state initialisation before each agent turn |
| 6 | **Full Simulation** | Complete 6-step workflow — runs without GCP credentials |
| 7 | **Real ADK Session** | Wiring everything to a live Vertex AI runner |
| 8 | **Structured Logging** | JSON logs compatible with Cloud Logging dashboards |

## Prerequisites
- Python 3.11+
- **Simulation sections**: no credentials needed
- **Real ADK sections**: a GCP project with Vertex AI API enabled

## Architecture Overview

### The Problem: Long-Running Workflows Break Stateless Chatbots

A new-hire onboarding spans **days or weeks** — document signing, hardware shipping, IT provisioning. A naïve chatbot approach dumps all history into the system prompt on every turn:

- Token bloat → expensive and slow
- Context pollution → LLM "hallucinates" which step it's on
- No pause/resume → container must stay alive the whole time

### The ADK Solution: Three Architectural Shifts

| Old Pattern | ADK Pattern | Why |
|-------------|-------------|-----|
| JSON chat log as memory | Enum state machine in SQLite | Grounded, no hallucination |
| Always-on polling | Event-driven dormancy (scale to 0) | Cost-efficient |
| Monolithic single agent | Root agent + sub-agent delegation | Focused, composable prompts |

### State Machine

```
  START
    │  (user provides name/email/start_date)
    ▼
  WELCOME_SENT  ◄─── IDLE GATE: agent sleeps, waits for webhook
    │  (document_signed webhook arrives → state_delta injected)
    ▼
  DOCUMENTS_SIGNED
    │  (root_agent transfers to it_agent)
    ▼
  IT_PROVISIONED  ◄── IDLE GATE: agent sleeps, waits for webhook
    │  (hardware_delivered webhook arrives → state_delta injected)
    ▼
  HARDWARE_DELIVERED
    │  (send_day_one_schedule tool called)
    ▼
  COMPLETED  (terminal)
```

### Component Map

```
app/
├── state_schema.py    ← OnboardingStep constants (6 states)
├── tools.py           ← 4 ADK tool functions (ToolContext pattern)
├── agent.py           ← root_agent + it_agent + before_agent_callback
├── resume_handler.py  ← webhook → state_delta → runner.run_async()
├── live_onboarding.py ← demo case management + HTML artifact generation
└── fast_api_app.py    ← FastAPI server (not covered in this notebook)
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Install Dependencies
# ─────────────────────────────────────────────────────────────────────────────
# google-adk        : Agent Development Kit — Agent, Runner, ToolContext, etc.
# google-cloud-aiplatform : Vertex AI integration & Agent Engine deployment
# google-genai      : Low-level genai types (Content, Part, HttpRetryOptions)
# nest_asyncio      : Allows asyncio.run() inside Jupyter/Colab event loops
# cloudpickle       : Required for Agent Engine serialisation

!pip install -q \
    google-adk>=1.0.0 \
    google-cloud-aiplatform \
    google-genai \
    nest_asyncio \
    cloudpickle

print('Dependencies installed.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: Logging Configuration
#
# We configure structured, coloured logging that mirrors the Cloud Logging
# JSON format used by OnboardingResumeHandler._log_structured() in production.
# Running this cell first ensures every subsequent cell emits rich logs.
# ─────────────────────────────────────────────────────────────────────────────

import logging
import sys
import json
from datetime import datetime


class _ColourFormatter(logging.Formatter):
    """ANSI-coloured formatter for Colab/terminal output."""

    _STYLES = {
        logging.DEBUG:    ("\x1b[38;21m",  "DEBUG  "),
        logging.INFO:     ("\x1b[34;1m",   "INFO   "),
        logging.WARNING:  ("\x1b[33;1m",   "WARN   "),
        logging.ERROR:    ("\x1b[31;1m",   "ERROR  "),
        logging.CRITICAL: ("\x1b[31;5m",   "CRIT   "),
    }
    _RESET = "\x1b[0m"

    def format(self, record: logging.LogRecord) -> str:
        colour, label = self._STYLES.get(record.levelno, (self._RESET, "      "))
        ts = datetime.fromtimestamp(record.created).strftime("%H:%M:%S.%f")[:-3]
        return f"{colour}{ts} {label}{self._RESET} [{record.name}] {record.getMessage()}"


def configure_logging(level: int = logging.DEBUG) -> logging.Logger:
    """
    Sets up the root logger with a coloured console handler.

    Calling this multiple times is safe (handlers are cleared first),
    which matters because Colab re-runs cells without restarting the kernel.

    Args:
        level: Minimum log level (default DEBUG to see everything).

    Returns:
        Logger named 'adk_onboarding' used throughout this notebook.
    """
    root = logging.getLogger()
    root.handlers.clear()  # Prevent duplicate handlers on cell re-run

    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(_ColourFormatter())
    root.addHandler(handler)
    root.setLevel(level)

    # Silence noisy third-party loggers
    for name in ("urllib3", "httpcore", "httpx", "google.auth", "grpc", "google.api_core"):
        logging.getLogger(name).setLevel(logging.WARNING)

    logger = logging.getLogger("adk_onboarding")
    logger.info("Logging configured | level=%s", logging.getLevelName(level))
    return logger


log = configure_logging()

# Apply nest_asyncio so async/await works inside notebook cells.
# Without this, asyncio.run() raises "This event loop is already running".
import nest_asyncio
nest_asyncio.apply()
log.info("nest_asyncio applied — async/await enabled in notebook cells")

## Google Cloud Authentication

**Simulation cells (Parts 1–5 below) need no credentials.**  
Real ADK cells (Part 6+) require Vertex AI access.

```python
# Option A — interactive Colab
from google.colab import auth
auth.authenticate_user()

# Option B — service account / ADC
# gcloud auth application-default login
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL: GCP Project Configuration
# ─────────────────────────────────────────────────────────────────────────────
import os

# ── User-configurable constants ───────────────────────────────────────────────
GCP_PROJECT_ID = "your-gcp-project-id"   # Replace with your project
GCP_LOCATION   = "us-central1"            # Vertex AI region

# The model used in the original source. Change to any available Gemini model:
#   gemini-2.0-flash, gemini-2.0-flash-lite, gemini-1.5-flash, etc.
GEMINI_MODEL   = "gemini-2.0-flash"

# ── Set environment variables expected by google-adk and google-genai ────────
os.environ["GOOGLE_CLOUD_PROJECT"]        = GCP_PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"]       = "global"   # ADK uses 'global', not a region
os.environ["GOOGLE_GENAI_USE_ENTERPRISE"] = "True"     # Route through Vertex, not AI Studio

log.info("GCP project=%s  location=%s  model=%s", GCP_PROJECT_ID, GCP_LOCATION, GEMINI_MODEL)

# ── Colab interactive auth (skipped silently outside Colab) ──────────────────
try:
    from google.colab import auth as _colab_auth
    _colab_auth.authenticate_user()
    log.info("Colab user authentication completed")
except ImportError:
    log.info("Not in Colab — using Application Default Credentials (ADC)")
except Exception as exc:
    log.warning("Colab auth skipped: %s", exc)

---
## Part 1 — State Schema: Grounded, Durable Memory

### Why Enums Beat Raw JSON

A typical chatbot stores memory like this:
```json
{"step": "after_welcome_before_docs", "history": [{...}, {...}, ...]}
```
The LLM must parse arbitrary JSON and reason about free-form strings — a recipe for hallucinations.

**ADK approach:** a small set of immutable string constants (like an enum) stored in SQLite:
```python
state["current_step"] = OnboardingStep.WELCOME_SENT
```

The agent's instruction template renders this directly: `Current Step: {current_step}`.  
The LLM sees a known token (`WELCOME_SENT`) and can reason about it reliably.

### Idle Gates
Two states are **idle gates** — the agent stops executing, the container scales to zero,  
and only a webhook can advance the state machine:

- `WELCOME_SENT` → waits for `document_signed` webhook  
- `IT_PROVISIONED` → waits for `hardware_delivered` webhook

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# MODULE: state_schema.py
# Source: app/state_schema.py
# ═════════════════════════════════════════════════════════════════════════════
#
# Design decisions:
#   - Plain class with string constants (not enum.Enum) so ADK's JSON session
#     serialiser can store and compare values without custom deserialisation.
#   - Immutable string constants prevent typos and enable IDE completion.
#   - Human-readable names appear clearly in Cloud Logs and agent prompts.

from typing import Optional

log_schema = logging.getLogger("adk_onboarding.state_schema")


class OnboardingStep:
    """
    State machine constants for the new-hire onboarding workflow.

    Linear progression:
        START → WELCOME_SENT → DOCUMENTS_SIGNED → IT_PROVISIONED
              → HARDWARE_DELIVERED → COMPLETED

    States with idle gates (agent pauses, waits for external webhook):
        WELCOME_SENT    — waits for 'document_signed'
        IT_PROVISIONED  — waits for 'hardware_delivered'
    """

    START              = "START"
    """Initial state. Agent collects new hire name/email/start_date."""

    WELCOME_SENT       = "WELCOME_SENT"
    """
    Welcome packet dispatched. IDLE GATE: execution pauses until
    the employee signs documents (triggered by 'document_signed' webhook).
    """

    DOCUMENTS_SIGNED   = "DOCUMENTS_SIGNED"
    """
    Employee countersigned. Agent delegates IT provisioning
    to the it_agent sub-agent.
    """

    IT_PROVISIONED     = "IT_PROVISIONED"
    """
    Corporate accounts (email, Slack) created. IDLE GATE: execution
    pauses until laptop is delivered ('hardware_delivered' webhook).
    """

    HARDWARE_DELIVERED = "HARDWARE_DELIVERED"
    """Laptop delivered. Agent sends the personalised Day-One schedule."""

    COMPLETED          = "COMPLETED"
    """Terminal state. All onboarding steps finished."""

    # ── Helpers (not present in original, added for notebook clarity) ─────────

    @classmethod
    def ordered(cls) -> list:
        """Returns all states in chronological order."""
        return [
            cls.START, cls.WELCOME_SENT, cls.DOCUMENTS_SIGNED,
            cls.IT_PROVISIONED, cls.HARDWARE_DELIVERED, cls.COMPLETED,
        ]

    @classmethod
    def is_idle_gate(cls, step: str) -> bool:
        """True if the agent should pause at this step and wait for a webhook."""
        return step in {cls.WELCOME_SENT, cls.IT_PROVISIONED}

    @classmethod
    def next_step(cls, current: str) -> Optional[str]:
        """Returns the next state, or None if already at COMPLETED."""
        steps = cls.ordered()
        try:
            idx = steps.index(current)
            return steps[idx + 1] if idx < len(steps) - 1 else None
        except ValueError:
            return None


# ── Visualise the state machine ───────────────────────────────────────────────
log_schema.info("OnboardingStep defined — %d states", len(OnboardingStep.ordered()))

print("\n  NEW-HIRE ONBOARDING STATE MACHINE\n")
for step in OnboardingStep.ordered():
    gate   = "  ◄─ IDLE GATE" if OnboardingStep.is_idle_gate(step) else ""
    nxt    = OnboardingStep.next_step(step)
    arrow  = f"  ──►  {nxt}" if nxt else "  (terminal)"
    print(f"  {step:25}{gate}")
    print(f"  {arrow}\n")

---
## Part 2 — ADK Tools: The `ToolContext` Pattern

### How ADK Tools Work

An ADK tool is a regular Python function. ADK:
1. Inspects the signature and docstring → generates a tool spec
2. Passes that spec to Gemini in the system prompt
3. When Gemini outputs a function call, ADK invokes the Python function
4. Returns the result dict back to Gemini's context

### The `ToolContext` Secret Parameter

```python
def my_tool(visible_arg: str, tool_context: ToolContext) -> dict:
    #                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    #  ADK injects this automatically. Gemini NEVER sees it.
    #  It gives the tool privileged access to mutable session state.
    tool_context.state["current_step"] = OnboardingStep.COMPLETED
    return {"status": "done"}
```

State changes inside a tool are **atomically persisted** to SQLite — they survive container restarts.

### Tool Design Rules Used in This Project

1. **Single responsibility** — each tool does one thing
2. **State advancement** — each tool advances `current_step` by exactly one step
3. **Signal bookkeeping** — tools update `pending_signals` to register expected webhooks
4. **Idempotency** — tools overwrite state, never append, so re-runs are safe

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# MODULE: tools.py
# Source: app/tools.py
# ═════════════════════════════════════════════════════════════════════════════
#
# Four ADK tool functions. In production these call real APIs:
#   send_welcome_packet       → DocuSign / Google Workspace API
#   provision_software_accounts → Google Admin SDK + Slack API
#   check_hardware_delivery   → FedEx/UPS tracking REST API
#   send_day_one_schedule     → Gmail API + Google Calendar API
#
# For the demo they return realistic mock data.
#
# The 'tool_context' parameter is annotated as Any here so this module
# works both with the real google.adk.tools.ToolContext and with the
# SimulatedToolContext defined below.

from typing import Any

log_tools = logging.getLogger("adk_onboarding.tools")


class SimulatedToolContext:
    """
    Drop-in replacement for google.adk.tools.ToolContext used in simulation cells.

    Holds a plain dict as session state so all tool functions can be exercised
    without a live ADK runner. In production, ADK creates the real ToolContext
    and injects it automatically — user code never constructs one manually.
    """

    def __init__(self, initial_state: dict = None):
        self.state = initial_state or {}
        log_tools.debug("SimulatedToolContext created | keys=%s", list(self.state.keys()))

    def __repr__(self):
        return f"SimulatedToolContext(step={self.state.get('current_step', '?')})"


# ─────────────────────────────────────────────────────────────────────────────
# TOOL 1: send_welcome_packet
# Triggered at: START → WELCOME_SENT
# ─────────────────────────────────────────────────────────────────────────────

def send_welcome_packet(
    name: str,
    email: str,
    start_date: str,
    tool_context: Any,
) -> dict:
    """
    Sends the initial welcome packet and document signature links to the new hire.

    This is the FIRST tool called in the workflow. It:
      1. Stores the new hire's profile in session state (persisted to SQLite).
      2. Advances current_step: START → WELCOME_SENT.
      3. Sets pending_signals=["document_signed"] to register the expected webhook.

    After this tool runs the agent enters an IDLE GATE — it will not call
    any more tools until the document_signed webhook wakes it.

    Args:
        name:         Full legal name of the new hire.
        email:        Personal email address (corporate email does not exist yet).
        start_date:   Official start date (YYYY-MM-DD).
        tool_context: Injected by ADK. Provides mutable session state.

    Returns:
        dict: status, confirmation message, and the document signature URL.
    """
    log_tools.info("send_welcome_packet | name=%s email=%s start_date=%s",
                   name, email, start_date)

    # 1. Persist the new hire profile — all subsequent tools read from this dict
    tool_context.state["new_hire_details"] = {
        "name": name, "email": email, "start_date": start_date,
    }
    log_tools.debug("new_hire_details persisted to session state")

    # 2. Advance the state machine
    prev = tool_context.state.get("current_step", "START")
    tool_context.state["current_step"] = OnboardingStep.WELCOME_SENT
    log_tools.info("State: %s → %s", prev, OnboardingStep.WELCOME_SENT)

    # 3. Register the webhook we expect next
    tool_context.state["pending_signals"] = ["document_signed"]
    log_tools.debug("pending_signals=%s", tool_context.state["pending_signals"])

    return {
        "status": "success",
        "message": f"Welcome packet sent to {name} ({email}). Documents pending signature.",
        "signature_link": f"https://onboarding.example.com/sign?email={email}",
    }


# ─────────────────────────────────────────────────────────────────────────────
# TOOL 2: provision_software_accounts
# Triggered at: DOCUMENTS_SIGNED → IT_PROVISIONED
# Called BY: it_agent (the sub-agent), not root_agent directly
# ─────────────────────────────────────────────────────────────────────────────

def provision_software_accounts(
    username: str,
    tool_context: Any,
) -> dict:
    """
    Provisions corporate software accounts (email, Slack) for the new hire.

    This tool belongs to the it_agent sub-agent, keeping IT provisioning
    logic separate from HR coordination. Separation of concerns:
      - root_agent handles HR steps (welcome, hardware check, schedule)
      - it_agent owns this single tool — focused prompt, minimal blast radius

    State changes:
      - Stores corporate_email inside new_hire_details (used by send_day_one_schedule)
      - Advances: DOCUMENTS_SIGNED → IT_PROVISIONED
      - Clears 'document_signed' signal, adds 'hardware_delivered'

    Args:
        username:     Desired corporate username prefix (e.g., 'j.smith').
        tool_context: Injected by ADK.

    Returns:
        dict: corporate_email, slack_user, and temporary_password.
    """
    log_tools.info("provision_software_accounts | username=%s", username)

    corporate_email = f"{username}@example.com"

    # Store corporate email for use by send_day_one_schedule
    details = tool_context.state.setdefault("new_hire_details", {})
    details["corporate_email"] = corporate_email
    log_tools.debug("Corporate email stored: %s", corporate_email)

    # Advance state machine
    prev = tool_context.state.get("current_step", "?")
    tool_context.state["current_step"] = OnboardingStep.IT_PROVISIONED
    log_tools.info("State: %s → %s", prev, OnboardingStep.IT_PROVISIONED)

    # Signal bookkeeping: clear previous signal, register next
    signals = tool_context.state.get("pending_signals", [])
    if "document_signed" in signals:
        signals.remove("document_signed")
    signals.append("hardware_delivered")
    tool_context.state["pending_signals"] = signals
    log_tools.debug("pending_signals=%s", signals)

    return {
        "status":             "success",
        "corporate_email":    corporate_email,
        "slack_user":         f"@{username}",
        "temporary_password": "TempPass2026!",  # In prod: Vault/Secret Manager ref
    }


# ─────────────────────────────────────────────────────────────────────────────
# TOOL 3: check_hardware_delivery
# Triggered at: IT_PROVISIONED → HARDWARE_DELIVERED
# ─────────────────────────────────────────────────────────────────────────────

def check_hardware_delivery(
    tracking_id: str,
    tool_context: Any,
) -> dict:
    """
    Checks delivery status of the new hire's laptop.

    Validation: tracking IDs must start with 'HW-' (e.g., HW-12345).
    This convention is taught to the agent via its instruction prompt so
    it can ask the user for the right format.

    In production: calls FedEx/UPS REST API with the tracking number.

    Args:
        tracking_id:  Shipment tracking number.
        tool_context: Injected by ADK.

    Returns:
        dict: 'delivered' or 'in_transit' status with carrier info.
    """
    log_tools.info("check_hardware_delivery | tracking_id=%s", tracking_id)

    if tracking_id.startswith("HW-"):
        prev = tool_context.state.get("current_step", "?")
        tool_context.state["current_step"] = OnboardingStep.HARDWARE_DELIVERED
        log_tools.info("State: %s → %s", prev, OnboardingStep.HARDWARE_DELIVERED)

        signals = tool_context.state.get("pending_signals", [])
        if "hardware_delivered" in signals:
            signals.remove("hardware_delivered")
        tool_context.state["pending_signals"] = signals

        recipient = (
            tool_context.state.get("new_hire_details", {})
                              .get("name", "Resident")
        )
        log_tools.info("Delivery confirmed | signed_by=%s", recipient)
        return {
            "status":      "delivered",
            "tracking_id": tracking_id,
            "carrier":     "FedEx",
            "signed_by":   recipient,
        }

    log_tools.info("Package in transit | tracking_id=%s", tracking_id)
    return {
        "status":      "in_transit",
        "tracking_id": tracking_id,
        "message":     "Package is in transit to employee residence.",
    }


# ─────────────────────────────────────────────────────────────────────────────
# TOOL 4: send_day_one_schedule
# Triggered at: HARDWARE_DELIVERED → COMPLETED (terminal)
# ─────────────────────────────────────────────────────────────────────────────

def send_day_one_schedule(
    email: str,
    tool_context: Any,
) -> dict:
    """
    Sends the personalised Day-One itinerary to the employee's corporate email.

    This is the FINAL tool — after it runs the state machine reaches COMPLETED
    and no further tools will be called.

    In production: sends HTML email via Gmail API and creates Calendar events.

    Args:
        email:        Corporate email address (created by provision_software_accounts).
        tool_context: Injected by ADK.

    Returns:
        dict: confirmation message and full itinerary list.
    """
    log_tools.info("send_day_one_schedule | email=%s", email)

    prev = tool_context.state.get("current_step", "?")
    tool_context.state["current_step"] = OnboardingStep.COMPLETED
    tool_context.state["pending_signals"] = []  # Terminal — no more webhooks
    log_tools.info("State: %s → %s (TERMINAL)", prev, OnboardingStep.COMPLETED)

    name = tool_context.state.get("new_hire_details", {}).get("name", "New Hire")

    itinerary = [
        "09:00 AM — Welcome & IT Login Setup",
        "10:00 AM — Meet the Manager & Team Introduction",
        "11:30 AM — Platform Architecture Overview",
        "01:00 PM — Lunch break",
        "02:00 PM — ADK Workflow Walkthrough",
        "03:30 PM — HR Policies & Benefits Overview",
        "04:30 PM — Q&A and Wrap-up",
    ]

    log_tools.info("Day-One schedule dispatched | recipient=%s | items=%d",
                   name, len(itinerary))
    return {
        "status":    "success",
        "message":   f"Day-One schedule sent to {name} at {email}.",
        "itinerary": itinerary,
    }


log_tools.info("Tools module loaded — 4 tools ready")
print("\nTools registered:")
for fn in [send_welcome_packet, provision_software_accounts,
           check_hardware_delivery, send_day_one_schedule]:
    args = ", ".join(
        p for p in fn.__code__.co_varnames[:fn.__code__.co_argcount]
        if p != "tool_context"
    )
    print(f"  {fn.__name__}({args})")

---
## Part 3 — Resume Handler: Waking a Sleeping Agent

### The Idle Gate Contract

When the agent reaches `WELCOME_SENT` it stops. No more tool calls until an external
system signals that documents were signed. This could be hours or days later.

The `OnboardingResumeHandler` provides two async methods — one per idle gate —
that external webhook endpoints call:

```
POST /webhooks/document_signed
    → receive_signed_documents_callback(user_id, session_id)
        → runner.run_async(..., state_delta={"current_step": DOCUMENTS_SIGNED})

POST /webhooks/hardware_delivered
    → receive_hardware_delivery_callback(user_id, session_id, tracking_id)
        → runner.run_async(..., state_delta={"current_step": HARDWARE_DELIVERED})
```

### The `state_delta` Parameter

`state_delta` atomically patches the stored session state **before** the agent
sees the new message. So when the agent wakes up, `{current_step}` in its instruction
already shows the new state — the agent doesn't have to infer the transition itself.

### Structured Logging for Production Observability

The handler uses `_log_structured()` which emits JSON lines that Cloud Logging
can index and query. You can build Monitoring alerts on:
- `event: "state_transition"` to trace every step
- `event: "runner_turn_failure"` to page on-call for broken workflows

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# MODULE: resume_handler.py
# Source: app/resume_handler.py
# ═════════════════════════════════════════════════════════════════════════════

import json as _json
import asyncio

log_handler = logging.getLogger("adk_onboarding.resume_handler")


class OnboardingResumeHandler:
    """
    Handles external webhook events that advance the onboarding state machine.

    Each method corresponds to one idle gate in the state machine:
      - receive_signed_documents_callback  → WELCOME_SENT gate
      - receive_hardware_delivery_callback → IT_PROVISIONED gate

    Both methods follow the same pattern:
      1. Log the incoming webhook event
      2. Log the state machine transition
      3. Call runner.run_async() with state_delta to atomically update state
         and resume agent execution
      4. Log every ADK event emitted during the wake turn
      5. Log success or failure with full error context

    The handler holds a reference to the ADK Runner so it can drive execution
    without needing a live HTTP connection from the original user session.
    This is the 'ambient execution' pattern — the agent runs in the background,
    not in direct response to a user message.
    """

    def __init__(self, runner):
        """
        Args:
            runner: An ADK Runner (or SimulatedRunner) bound to the active App.
        """
        self.runner = runner
        log_handler.info("OnboardingResumeHandler initialised | runner=%s",
                         type(runner).__name__)

    def _log_structured(self, severity: str, message: str, **kwargs) -> None:
        """
        Emits a JSON-formatted log line compatible with Google Cloud Logging.

        Cloud Logging parses structured JSON and indexes every key as a field,
        enabling precise log-based metrics and alerts.

        Example output (prettified):
        {
            "severity": "INFO",
            "message": "State machine transitioned to DOCUMENTS_SIGNED",
            "event": "state_transition",
            "session_id": "abc-123",
            "new_step": "DOCUMENTS_SIGNED"
        }

        Args:
            severity: Cloud Logging severity string ("INFO", "ERROR", etc.).
            message:  Human-readable log message.
            **kwargs: Additional structured fields (event, session_id, etc.).
        """
        payload = {"severity": severity, "message": message, **kwargs}
        # In production this line goes to Cloud Logging via the GCP log handler.
        # In this notebook we also echo through the standard logger.
        level = getattr(logging, severity, logging.INFO)
        log_handler.log(level, _json.dumps(payload))

    async def receive_signed_documents_callback(
        self, user_id: str, session_id: str
    ) -> None:
        """
        Simulates an external webhook notifying that the employee signed documents.

        Flow:
          1. Log the incoming webhook
          2. Call runner.run_async() with:
             - A new user message: "Resume onboarding: Contract has been signed."
             - state_delta: current_step → DOCUMENTS_SIGNED, pending_signals → []
          3. Stream and log every ADK event from the wake turn
          4. Log success or raise on failure

        Args:
            user_id:    The ADK user ID for this onboarding session.
            session_id: The ADK session ID (matches the paused session).
        """
        self._log_structured(
            severity="INFO",
            message=f"document_signed webhook received for session {session_id}",
            event="webhook_received",
            webhook_type="document_signed",
            session_id=session_id,
            user_id=user_id,
        )

        try:
            self._log_structured(
                severity="INFO",
                message=f"Transitioning to {OnboardingStep.DOCUMENTS_SIGNED}",
                event="state_transition",
                session_id=session_id,
                new_step=OnboardingStep.DOCUMENTS_SIGNED,
            )

            # runner.run_async() hydrates the paused session, applies state_delta
            # atomically, then re-runs the agent as if a new user message arrived.
            # The agent sees current_step=DOCUMENTS_SIGNED and proceeds to step 3.
            async for event in self.runner.run_async(
                user_id=user_id,
                session_id=session_id,
                new_message="Resume onboarding: Contract has been signed.",
                state_delta={
                    "current_step":   OnboardingStep.DOCUMENTS_SIGNED,
                    "pending_signals": [],
                },
            ):
                # Each 'event' is an ADK RunEvent (tool call, model response, etc.)
                self._log_structured(
                    severity="INFO",
                    message=f"Wake-up event: {event}",
                    event="runner_event",
                    session_id=session_id,
                )

            self._log_structured(
                severity="INFO",
                message="Document signature wake turn completed",
                event="runner_turn_success",
                session_id=session_id,
            )

        except Exception as exc:
            self._log_structured(
                severity="ERROR",
                message=f"Document signature wake turn failed: {exc!s}",
                event="runner_turn_failure",
                session_id=session_id,
                error=str(exc),
            )
            raise

    async def receive_hardware_delivery_callback(
        self, user_id: str, session_id: str, tracking_id: str
    ) -> None:
        """
        Simulates a carrier webhook confirming laptop delivery.

        Identical pattern to receive_signed_documents_callback but:
          - Triggered by tracking_id arrival from the shipping carrier API
          - Advances to HARDWARE_DELIVERED instead of DOCUMENTS_SIGNED
          - Passes tracking_id in the resume message so the agent can
            call check_hardware_delivery() with the correct ID

        Args:
            user_id:     ADK user ID.
            session_id:  ADK session ID.
            tracking_id: Carrier tracking number (e.g., HW-55443).
        """
        self._log_structured(
            severity="INFO",
            message=f"hardware_delivered webhook for tracking {tracking_id}",
            event="webhook_received",
            webhook_type="hardware_delivered",
            session_id=session_id,
            tracking_id=tracking_id,
        )

        try:
            self._log_structured(
                severity="INFO",
                message=f"Transitioning to {OnboardingStep.HARDWARE_DELIVERED}",
                event="state_transition",
                session_id=session_id,
                new_step=OnboardingStep.HARDWARE_DELIVERED,
            )

            async for event in self.runner.run_async(
                user_id=user_id,
                session_id=session_id,
                new_message=(
                    f"Resume onboarding: Hardware delivered with "
                    f"tracking ID {tracking_id}."
                ),
                state_delta={
                    "current_step":   OnboardingStep.HARDWARE_DELIVERED,
                    "pending_signals": [],
                },
            ):
                self._log_structured(
                    severity="INFO",
                    message=f"Wake-up event: {event}",
                    event="runner_event",
                    session_id=session_id,
                )

            self._log_structured(
                severity="INFO",
                message="Hardware delivery wake turn completed",
                event="runner_turn_success",
                session_id=session_id,
            )

        except Exception as exc:
            self._log_structured(
                severity="ERROR",
                message=f"Hardware delivery wake turn failed: {exc!s}",
                event="runner_turn_failure",
                session_id=session_id,
                error=str(exc),
            )
            raise


log_handler.info("OnboardingResumeHandler class defined")

---
## Part 4 — Agent Construction: Root Agent, Sub-Agent, Callback

### The Agent Instruction Template

The instruction is a Jinja-style template. ADK fills in the variables from session state
**before every turn**, giving the agent grounded, up-to-date context:

```
Current Step: {current_step}        ← renders as e.g. "WELCOME_SENT"
New Hire Details: {new_hire_details} ← renders as the stored dict
Pending Signals: {pending_signals}   ← renders as e.g. ["document_signed"]
```

This replaces the need for a long chat history — the agent only needs to see its
current state, not how it got there.

### Before-Agent Callback

`initialize_onboarding_state` runs **once before the first turn** (and every subsequent
turn). It safely initialises the three state keys so the template variables are never
undefined — even if the session was just created.

### Sub-Agent Delegation

When `current_step == DOCUMENTS_SIGNED`, the root agent's instruction says:  
*"Delegate to 'it_agent'. Do not call tools directly for provisioning accounts."*

ADK interprets this as a **transfer**: it_agent takes control, calls
`provision_software_accounts`, then transfers back to the root agent.

```python
root_agent = Agent(
    sub_agents=[it_agent],   # Makes it_agent available as a transfer target
    ...
)
```

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# MODULE: agent.py
# Source: app/agent.py
# ═════════════════════════════════════════════════════════════════════════════
#
# This cell requires google-adk to be installed and imports real ADK classes.
# If the import fails (e.g. package not installed yet), the simulation in
# Part 5 still works — it uses SimulatedToolContext and a mock runner.

try:
    from google.adk.agents import Agent
    from google.adk.agents.callback_context import CallbackContext
    from google.adk.apps import App
    from google.adk.models import Gemini
    from google.genai import types as genai_types
    ADK_AVAILABLE = True
    log.info("google-adk imported successfully")
except ImportError as exc:
    ADK_AVAILABLE = False
    log.warning("google-adk not available (%s) — Real ADK cells will be skipped", exc)


# ─────────────────────────────────────────────────────────────────────────────
# Before-agent callback
# ─────────────────────────────────────────────────────────────────────────────

if ADK_AVAILABLE:
    async def initialize_onboarding_state(callback_context: CallbackContext) -> None:
        """
        Ensures all three state machine keys exist before the agent renders
        its instruction template. Prevents KeyError if the session was just
        created and the keys have not been set yet.

        ADK calls this function once per turn, before the agent processes
        the incoming message. It is idempotent — if keys already exist,
        the existing values are left unchanged.

        Args:
            callback_context: Provided by ADK. Exposes mutable session state.
        """
        state = callback_context.state
        log.debug("before_agent_callback: checking state keys")

        if "current_step" not in state:
            state["current_step"] = OnboardingStep.START
            log.info("State initialised: current_step=%s", OnboardingStep.START)

        if "new_hire_details" not in state:
            state["new_hire_details"] = {}
            log.debug("State initialised: new_hire_details={}")

        if "pending_signals" not in state:
            state["pending_signals"] = []
            log.debug("State initialised: pending_signals=[]")


# ─────────────────────────────────────────────────────────────────────────────
# Agent instruction templates
# ─────────────────────────────────────────────────────────────────────────────

ROOT_AGENT_INSTRUCTION = """\
You are an HR Onboarding Coordinator Agent. Your goal is to safely guide the
onboarding process of new hires through a sequence of checkpoint steps.

Current Step: {current_step}
New Hire Details: {new_hire_details}
Pending Signals: {pending_signals}

Follow this state machine flow exactly:

1. START: Ask for the new hire's name, email, and start date.
   Once provided, invoke 'send_welcome_packet'.

2. WELCOME_SENT: Inform the user you are in an idle-time pause waiting for
   the employee to sign documents. Do not call other tools.

3. DOCUMENTS_SIGNED: Delegate IT account provisioning to the 'it_agent'
   sub-agent. Do NOT call provisioning tools directly.

4. IT_PROVISIONED: Ask for the hardware tracking ID (e.g. HW-12345) to
   check laptop shipping status. Once provided, invoke 'check_hardware_delivery'.

5. HARDWARE_DELIVERED: Invoke 'send_day_one_schedule' using the new hire's
   corporate email (stored in new_hire_details).

6. COMPLETED: Congratulate the user and list the day-one schedule.

Always stay grounded in your tools and current state. Do not skip steps or
invent details.
"""

IT_AGENT_INSTRUCTION = """\
You are an IT Provisioning Agent. Your goal is to provision corporate software
accounts (email, Slack) for the new hire.

Current Step: {current_step}
New Hire Details: {new_hire_details}

Instructions:
1. Ask the coordinator for the desired corporate username prefix if not provided.
2. Once provided, invoke 'provision_software_accounts'.
3. After provisioning, inform the coordinator that accounts are ready and
   transfer execution back to the parent HR coordinator agent.
"""

log.info("Agent instruction templates defined")


# ─────────────────────────────────────────────────────────────────────────────
# Build the agents (requires ADK + valid GCP credentials)
# ─────────────────────────────────────────────────────────────────────────────

if ADK_AVAILABLE:
    # it_agent: a focused specialist with a single tool
    # The root agent will transfer control here at step DOCUMENTS_SIGNED.
    it_agent = Agent(
        name="it_agent",
        model=Gemini(
            model=GEMINI_MODEL,
            # Retry transient API errors up to 3 times before failing
            retry_options=genai_types.HttpRetryOptions(attempts=3),
        ),
        instruction=IT_AGENT_INSTRUCTION,
        tools=[provision_software_accounts],  # Only one tool — tight scope
    )
    log.info("it_agent created | model=%s | tools=[provision_software_accounts]",
             GEMINI_MODEL)

    # root_agent: the HR coordinator that orchestrates the full workflow
    root_agent = Agent(
        name="hr_onboarding_coordinator",
        model=Gemini(
            model=GEMINI_MODEL,
            retry_options=genai_types.HttpRetryOptions(attempts=3),
        ),
        instruction=ROOT_AGENT_INSTRUCTION,
        tools=[
            send_welcome_packet,
            check_hardware_delivery,
            send_day_one_schedule,
            # Note: provision_software_accounts is NOT here — it's on it_agent
        ],
        sub_agents=[it_agent],          # Registers it_agent as a transfer target
        before_agent_callback=initialize_onboarding_state,
    )
    log.info("root_agent created | name=hr_onboarding_coordinator | sub_agents=[it_agent]")

    # App wraps root_agent and manages session persistence (SQLite by default)
    adk_app = App(root_agent=root_agent, name="app")
    log.info("ADK App created")
else:
    log.warning("Skipped agent construction — ADK not available")
    log.info("Proceed to Part 5 for the full simulation that runs without ADK")

---
## Part 5 — Full Workflow Simulation (No GCP Needed)

This section runs the complete 6-step workflow using only Python — no Gemini API calls,
no GCP credentials. A `SimulatedRunner` replaces the real ADK runner and calls
tools directly with a `SimulatedToolContext`.

**What this demonstrates:**
- The state machine transitions step by step
- How tools mutate shared session state
- How idle gates pause execution until a webhook fires
- The complete log trail from START to COMPLETED

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Simulated Runner: drives the workflow without a real LLM
# ─────────────────────────────────────────────────────────────────────────────

log_sim = logging.getLogger("adk_onboarding.simulation")


class SimulatedRunner:
    """
    Minimal stand-in for the ADK Runner used in resume_handler callbacks.

    Instead of calling Gemini and dispatching tool calls, it applies the
    state_delta directly to the shared SimulatedToolContext and yields a
    single descriptive event string — enough to exercise and demonstrate
    the OnboardingResumeHandler's logic without any network calls.

    In production the real Runner:
      1. Loads the session from SQLite
      2. Applies state_delta atomically
      3. Sends the new_message to Gemini
      4. Executes any tool calls Gemini requests
      5. Yields RunEvent objects for each step
    """

    def __init__(self, ctx: SimulatedToolContext):
        self._ctx = ctx

    async def run_async(self, user_id, session_id, new_message, state_delta=None):
        """
        Simulates a single agent turn.

        Applies state_delta, then yields one event string so the caller's
        async-for loop can iterate at least once (matching the real Runner API).
        """
        log_sim.debug("SimulatedRunner.run_async | session=%s | delta=%s",
                      session_id, state_delta)

        # Atomically apply the state patch — mirrors runner behaviour
        if state_delta:
            self._ctx.state.update(state_delta)
            log_sim.info("state_delta applied | keys=%s", list(state_delta.keys()))

        yield f"SimulatedEvent[session={session_id}, msg={new_message!r}]"


# ─────────────────────────────────────────────────────────────────────────────
# Simulate the complete onboarding workflow end-to-end
# ─────────────────────────────────────────────────────────────────────────────

async def run_full_simulation():
    """
    Drives the new-hire onboarding workflow through all 6 states.

    Sequence:
      1. HR collects new hire info → send_welcome_packet → WELCOME_SENT
      2. IDLE GATE: wait for document_signed webhook
      3. document_signed webhook → DOCUMENTS_SIGNED
      4. it_agent provisions accounts → IT_PROVISIONED
      5. IDLE GATE: wait for hardware_delivered webhook
      6. hardware_delivered webhook → HARDWARE_DELIVERED
      7. send_day_one_schedule → COMPLETED
    """

    log_sim.info("=" * 60)
    log_sim.info("SIMULATION START")
    log_sim.info("=" * 60)

    # Shared session state (persisted to SQLite in production)
    ctx = SimulatedToolContext({
        "current_step":   OnboardingStep.START,
        "new_hire_details": {},
        "pending_signals": [],
    })

    runner  = SimulatedRunner(ctx)
    handler = OnboardingResumeHandler(runner)

    SESSION_ID = "sim-session-001"
    USER_ID    = "hr-coordinator"

    # ── STEP 1: HR agent gathers info and sends welcome packet ─────────────────
    log_sim.info("--- STEP 1: Collect new hire info and send welcome packet ---")
    result = send_welcome_packet(
        name="Olivia Bennett",
        email="olivia.bennett@personal.com",
        start_date="2026-06-01",
        tool_context=ctx,
    )
    log_sim.info("Tool result: %s", result)
    assert ctx.state["current_step"] == OnboardingStep.WELCOME_SENT
    assert ctx.state["pending_signals"] == ["document_signed"]

    # ── IDLE GATE 1: agent is parked, container can scale to zero ──────────────
    log_sim.info("--- IDLE GATE: Agent parked at WELCOME_SENT ---")
    log_sim.info("    (In production: container scales to zero here)")
    log_sim.info("    (Waiting for: %s)", ctx.state["pending_signals"])

    # ── STEP 2: document_signed webhook arrives ────────────────────────────────
    log_sim.info("--- STEP 2: document_signed webhook fires ---")
    await handler.receive_signed_documents_callback(
        user_id=USER_ID,
        session_id=SESSION_ID,
    )
    assert ctx.state["current_step"] == OnboardingStep.DOCUMENTS_SIGNED

    # ── STEP 3: it_agent provisions corporate accounts ─────────────────────────
    log_sim.info("--- STEP 3: it_agent provisions software accounts ---")
    result = provision_software_accounts(
        username="o.bennett",
        tool_context=ctx,
    )
    log_sim.info("Tool result: %s", result)
    assert ctx.state["current_step"] == OnboardingStep.IT_PROVISIONED
    assert ctx.state["new_hire_details"]["corporate_email"] == "o.bennett@example.com"

    # ── IDLE GATE 2: waiting for hardware delivery ─────────────────────────────
    log_sim.info("--- IDLE GATE: Agent parked at IT_PROVISIONED ---")
    log_sim.info("    (Waiting for: %s)", ctx.state["pending_signals"])

    # ── STEP 4: hardware_delivered webhook fires ───────────────────────────────
    log_sim.info("--- STEP 4: hardware_delivered webhook fires ---")
    await handler.receive_hardware_delivery_callback(
        user_id=USER_ID,
        session_id=SESSION_ID,
        tracking_id="HW-55443",
    )
    assert ctx.state["current_step"] == OnboardingStep.HARDWARE_DELIVERED

    # ── STEP 5: check delivery (called by agent after webhook wakes it) ────────
    log_sim.info("--- STEP 5: Agent calls check_hardware_delivery ---")
    result = check_hardware_delivery(
        tracking_id="HW-55443",
        tool_context=ctx,
    )
    log_sim.info("Tool result: %s", result)
    # Note: state was already advanced to HARDWARE_DELIVERED by the webhook;
    # calling the tool again is idempotent — it stays at HARDWARE_DELIVERED.

    # ── STEP 6: send Day-One schedule → COMPLETED ─────────────────────────────
    log_sim.info("--- STEP 6: send_day_one_schedule ---")
    corporate_email = ctx.state["new_hire_details"]["corporate_email"]
    result = send_day_one_schedule(email=corporate_email, tool_context=ctx)
    log_sim.info("Tool result: %s", result)
    assert ctx.state["current_step"] == OnboardingStep.COMPLETED
    assert ctx.state["pending_signals"] == []

    log_sim.info("=" * 60)
    log_sim.info("SIMULATION COMPLETE — final state: %s",
                 ctx.state["current_step"])
    log_sim.info("=" * 60)

    return ctx.state


# Run the simulation
final_state = asyncio.run(run_full_simulation())

print("\n Final session state:")
for k, v in final_state.items():
    print(f"   {k}: {v}")

---
## Part 6 — Real ADK Session (Requires GCP)

This section runs the agent with a live Gemini model via Vertex AI.  
Ensure you have:
- A valid GCP project in `GCP_PROJECT_ID` (Cell 4)
- Vertex AI API enabled: `gcloud services enable aiplatform.googleapis.com`
- Application Default Credentials or Colab auth completed

The cell will be skipped gracefully if ADK is not available.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# REAL ADK SESSION
# Requires: ADK installed, GCP credentials, Vertex AI API enabled
# ─────────────────────────────────────────────────────────────────────────────

if not ADK_AVAILABLE:
    log.warning("Skipping real ADK session — google-adk not available")
else:
    from google.adk.runners import Runner
    from google.adk.sessions import InMemorySessionService
    from google.genai import types as genai_types

    log.info("Creating ADK Runner with InMemorySessionService")

    # InMemorySessionService: stores sessions in memory (no SQLite needed for demo).
    # In production use DatabaseSessionService (SQLite/Postgres) for durability.
    session_service = InMemorySessionService()

    # The Runner is the execution engine. It:
    #   - Manages session state (via session_service)
    #   - Routes user messages to the correct agent
    #   - Dispatches tool calls and feeds results back to Gemini
    #   - Yields RunEvent objects for each step of the turn
    adk_runner = Runner(
        app=adk_app,
        session_service=session_service,
    )
    log.info("ADK Runner created")

    # Create a new session for the demo user
    DEMO_USER_ID    = "demo-hr-coordinator"
    DEMO_SESSION_ID = "demo-session-001"

    async def start_real_session():
        """Creates a session and sends the opening message to the HR agent."""

        log.info("Creating session | user=%s session=%s",
                 DEMO_USER_ID, DEMO_SESSION_ID)

        await session_service.create_session(
            app_name="app",
            user_id=DEMO_USER_ID,
            session_id=DEMO_SESSION_ID,
        )

        # Send the opening user message — the agent's before_agent_callback runs
        # first to initialise state, then the agent processes the message.
        opening_message = genai_types.Content(
            role="user",
            parts=[genai_types.Part.from_text(
                "Hello, I'd like to start onboarding a new employee: "
                "Olivia Bennett, olivia@personal.com, starting 2026-06-01."
            )],
        )

        log.info("Sending opening message to root_agent")

        async for event in adk_runner.run_async(
            user_id=DEMO_USER_ID,
            session_id=DEMO_SESSION_ID,
            new_message=opening_message,
        ):
            # RunEvent types include: model_response, tool_call, tool_result,
            # agent_transfer, turn_complete, etc.
            log.info("ADK event: %s", event)

        # Inspect state after the turn
        session = await session_service.get_session(
            app_name="app",
            user_id=DEMO_USER_ID,
            session_id=DEMO_SESSION_ID,
        )
        log.info("Session state after turn: current_step=%s",
                 session.state.get("current_step"))
        return session

    try:
        demo_session = asyncio.run(start_real_session())
        print("\nSession state after first turn:")
        for k, v in demo_session.state.items():
            print(f"  {k}: {v}")
    except Exception as exc:
        log.error("Real ADK session failed: %s", exc)
        log.info("Check GCP_PROJECT_ID, credentials, and Vertex AI API access")

---
## Part 7 — Triggering Webhooks on a Real ADK Session

After the real ADK session reaches `WELCOME_SENT`, the agent is idle.
Run this cell to simulate the document-signed webhook and wake it.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# WEBHOOK SIMULATION — wake the parked real ADK session
# ─────────────────────────────────────────────────────────────────────────────

if not ADK_AVAILABLE:
    log.warning("Skipping — ADK not available")
else:
    async def fire_document_signed_webhook():
        """
        Fires the document_signed webhook against the live ADK session.

        In production, this function is called by a FastAPI endpoint:

            @app.post("/webhooks/document_signed")
            async def document_signed_webhook(payload: WebhookPayload):
                await resume_handler.receive_signed_documents_callback(
                    user_id=payload.user_id,
                    session_id=payload.session_id,
                )

        DocuSign/HelloSign POSTs to that endpoint after the employee signs.
        The endpoint calls the resume handler, which wakes the ADK session.
        """
        real_handler = OnboardingResumeHandler(adk_runner)

        log.info("Firing document_signed webhook for session=%s", DEMO_SESSION_ID)
        await real_handler.receive_signed_documents_callback(
            user_id=DEMO_USER_ID,
            session_id=DEMO_SESSION_ID,
        )

        # Inspect state after the wake turn
        session = await session_service.get_session(
            app_name="app",
            user_id=DEMO_USER_ID,
            session_id=DEMO_SESSION_ID,
        )
        log.info("Session state after webhook: current_step=%s",
                 session.state.get("current_step"))
        return session

    try:
        session_after_webhook = asyncio.run(fire_document_signed_webhook())
        print(f"\ncurrent_step after webhook: "
              f"{session_after_webhook.state.get('current_step')}")
    except Exception as exc:
        log.error("Webhook simulation failed: %s", exc)

---
## Part 8 — State Inspection and Debugging

These utilities help you introspect session state during development.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# State inspection utilities
# ─────────────────────────────────────────────────────────────────────────────

def print_state_snapshot(state: dict, title: str = "Session State") -> None:
    """
    Pretty-prints a session state dict with context about each key.

    Useful for debugging agent behaviour — call after any tool or webhook
    to see exactly what's persisted in the session.

    Args:
        state: The session state dict (from SimulatedToolContext.state
               or real ADK session.state).
        title: Header label for the printout.
    """
    step = state.get("current_step", "UNKNOWN")
    is_idle = OnboardingStep.is_idle_gate(step)
    is_done = step == OnboardingStep.COMPLETED

    print(f"\n╔══ {title} " + "═" * max(0, 50 - len(title)) + "╗")
    print(f"║  current_step   : {step}")
    if is_idle:
        print(f"║                   ^ IDLE GATE — waiting for webhook")
    if is_done:
        print(f"║                   ^ TERMINAL — workflow complete")
    print(f"║  pending_signals: {state.get('pending_signals', [])}")

    details = state.get("new_hire_details", {})
    if details:
        print(f"║  new_hire_details:")
        for k, v in details.items():
            print(f"║    {k:<22}: {v}")
    else:
        print(f"║  new_hire_details: (empty)")
    print("╚" + "═" * 53 + "╝\n")


def simulate_step_by_step() -> None:
    """
    Runs each tool individually, printing a state snapshot after each call.
    Useful as a visual walkthrough of the state machine for learning purposes.
    """
    ctx = SimulatedToolContext({
        "current_step": OnboardingStep.START,
        "new_hire_details": {},
        "pending_signals": [],
    })

    print_state_snapshot(ctx.state, "Initial State")

    send_welcome_packet("Alex Chen", "alex@personal.com", "2026-07-01", ctx)
    print_state_snapshot(ctx.state, "After send_welcome_packet")

    # Simulate the document_signed webhook state_delta
    ctx.state["current_step"]   = OnboardingStep.DOCUMENTS_SIGNED
    ctx.state["pending_signals"] = []
    print_state_snapshot(ctx.state, "After document_signed webhook")

    provision_software_accounts("a.chen", ctx)
    print_state_snapshot(ctx.state, "After provision_software_accounts")

    # Simulate the hardware_delivered webhook state_delta
    ctx.state["current_step"]   = OnboardingStep.HARDWARE_DELIVERED
    ctx.state["pending_signals"] = []
    print_state_snapshot(ctx.state, "After hardware_delivered webhook")

    send_day_one_schedule(
        ctx.state["new_hire_details"]["corporate_email"], ctx
    )
    print_state_snapshot(ctx.state, "Final State — COMPLETED")


simulate_step_by_step()

---
## Part 9 — Cloud Deployment (Vertex AI Agent Engine)

The original project deploys to **Vertex AI Agent Engine** (formerly Reasoning Engine),
which provides:
- Managed container lifecycle (auto-scaling, including scale-to-zero)
- Built-in session persistence (Cloud Spanner or Firestore)
- OpenTelemetry tracing out of the box
- HTTPS endpoints for webhook delivery

```
Local dev  →  uv run uvicorn app.fast_api_app:app --port 8000
Production →  Agent Engine (Vertex AI)
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CLOUD DEPLOYMENT: Vertex AI Agent Engine
# This cell shows the deployment pattern; it does NOT execute a real deployment.
# ─────────────────────────────────────────────────────────────────────────────

DEPLOY = False  # Set to True and fill in project details to run a real deployment

if DEPLOY and ADK_AVAILABLE:
    import vertexai
    from vertexai import agent_engines

    # Initialise the Vertex AI SDK with your project and region.
    # Agent Engine is only available in select regions; us-central1 is default.
    vertexai.init(
        project=GCP_PROJECT_ID,
        location=GCP_LOCATION,
        staging_bucket=f"gs://{GCP_PROJECT_ID}-adk-staging",
    )

    # agent_engines.create() packages the App into a container and deploys it.
    # The App must be importable as 'app.agent:app'.
    # Requirements are pulled from pyproject.toml / requirements.txt.
    remote_app = agent_engines.create(
        adk_app,
        requirements=[
            "google-adk>=1.0.0",
            "google-cloud-aiplatform",
            "google-genai",
        ],
        # extra_packages lists local Python packages to bundle into the image
        extra_packages=["app"],
    )

    log.info("Deployed to Agent Engine | resource_name=%s",
             remote_app.resource_name)

    # After deployment, query the remote agent just like the local runner:
    #   remote_app.query(input="Hello...", user_id="...", session_id="...")

else:
    print("Deployment skipped (DEPLOY=False).")
    print()
    print("To deploy, set DEPLOY=True and ensure:")
    print("  1. GCP_PROJECT_ID is your real project")
    print("  2. Vertex AI API is enabled")
    print("  3. A staging GCS bucket exists:")
    print(f"       gs://{GCP_PROJECT_ID}-adk-staging")
    print("  4. Run: gcloud services enable aiplatform.googleapis.com")

---
## Summary & Key Takeaways

| Concept | Pattern Used | File |
|---------|-------------|------|
| Durable memory | Enum state machine in SQLite (not chat history) | `state_schema.py` |
| Tool-to-state binding | `ToolContext.state` mutation inside tool functions | `tools.py` |
| Idle gates | Agent pauses at `WELCOME_SENT` / `IT_PROVISIONED` | `agent.py` (instruction) |
| Webhook resumption | `runner.run_async(state_delta=...)` | `resume_handler.py` |
| Sub-agent delegation | Root agent transfers to `it_agent` for IT work | `agent.py` |
| State initialisation | `before_agent_callback` runs once per turn | `agent.py` |
| Production observability | Structured JSON logs → Cloud Logging | `resume_handler.py` |

### Next Steps

- **Run the real ADK session** (Part 6) once you have GCP credentials
- **Add a new step** — try inserting a `BACKGROUND_CHECK` state between `DOCUMENTS_SIGNED` and `IT_PROVISIONED`
- **Add a real tool** — replace `provision_software_accounts` with a real Google Admin SDK call
- **Deploy** — set `DEPLOY=True` in Part 9 and push to Agent Engine
- **Explore the frontend** — the original repo includes a React dashboard at `frontend/live-onboarding/`

**Source repo**: https://github.com/GoogleCloudPlatform/generative-ai/tree/main/agents/adk/new-hire-onboarding